--------------------------------------------------------------------------------------------------------------------

**This code uses the following dataset:**

    - Thredd's UCAR ERA5 Reanalysis data on a .25 Degree Latitude/Longitude Grid:
      https://thredds.rda.ucar.edu/thredds/catalog/files/g/d633000/e5.oper.an.pl/202112/catalog.html
  
--------------------------------------------------------------------------------------------------------------------

## ERA5 Potential Vorticity and Geopotential Heights

This notebook computes and plots **Potential Vorticity (PV)** and **geopotential height** at a selected pressure level from locally stored ERA5 GRIB files. PV is a powerful tool for identifying upper-level disturbances, jet stream structure, and stratospheric intrusions.

### Physical Background

**Potential Vorticity** (PV) combines the effects of absolute vorticity and static stability into a single conserved quantity:

$$\text{PV} = -g \left( f + \zeta \right) \frac{\partial \theta}{\partial p}$$

PV is expressed in **PVU** (Potential Vorticity Units), where 1 PVU = 10^-6 m^2 s^-1 K kg^-1. The **2 PVU surface** is the dynamical definition of the tropopause, and values above 2 PVU indicate stratospheric air. Elevated PV at upper levels marks tropopause folds and upper-level troughs that provide large-scale forcing for surface cyclogenesis and winter precipitation events.

**Geopotential height (Z)** is the height of a constant pressure surface above sea level, contoured in decameters (dam). Troughs and ridges organize the flow and steer surface weather systems.

In [ ]:
import os
from pathlib import Path
import shutil
import tempfile

# redirect cfgrib index files to home directory
index_dir = Path("/system/home/.cfgrib_indexes")
index_dir.mkdir(exist_ok=True)
os.environ["CFGRIB_INDEX_PATH"] = str(index_dir)

import xarray as xr
import numpy as np
from datetime import datetime as dt
from metpy.units import units
from metpy import calc as mpcalc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import cfgrib # noqa: F401 — imported to register the cfgrib xarray backend

## 1.) ERA5 GRIB Data Loader

`get_era5_ds` loads a single pressure-level variable from the read-only ERA5 archive by:

1. Locating the GRIB file at `ERA5_ROOT / variable / variable_YEAR.grib`
2. Copying it to a temporary directory cfgrib can write index files into
3. Opening with `xr.open_dataset(..., engine='cfgrib')`, at the requested level
4. Force-loading into memory with `.load()`, then deleting the temp copy

The `finally` block guarantees cleanup even if the dataset open fails.

In [ ]:
# loads a single pressure-level variable from a local GRIB file because /data1/ERA5 is read-only, each file is copied to a temp directory
# ERA5 local root
era5_root = Path("/system/ERA5data")

def get_era5_ds(variable: str, year: int, level: int):
    filepath = era5_root / variable / f"{variable}_{year}.grib"
    
    if not filepath.exists():
        raise FileNotFoundError(f"No GRIB file found at {filepath}")
    
    # copy to a temp dir you own so cfgrib can write its index file
    tmp_dir = Path(tempfile.mkdtemp(dir="/system/home/.cfgrib_indexes"))
    tmp_file = tmp_dir / filepath.name
    shutil.copy2(filepath, tmp_file)
    
    ds = xr.open_dataset(
        tmp_file,
        engine='cfgrib',
        filter_by_keys={'typeOfLevel': 'isobaricInhPa', 'level': level},
        chunks={'time': 24}
    )
    return ds

## 2.) User Configuration

All run parameters are defined here. No edits should be needed below this cell.

**Extent options:**
- Inner 48 states: `lonW, lonE, latS, latN = -134, -59, 23, 50`
- Including Alaska *(current)*: `lonW, lonE, latS, latN = -180, -30, 0, 70`

**Note:** Coordinates are padded by +-5deg when slicing ERA5 data to avoid edge artifacts in the contour plots near the map boundary.

In [ ]:
# specify vertical level, date/time
vlevel = 300
levelStr = str(vlevel)
Year, Month, Day, Hour, Minute = 2011, 2, 2, 0, 0
dateTime = dt(Year, Month, Day, Hour, Minute)
timeStr = dateTime.strftime("%Y-%m-%d %H%M UTC")

# data dimensions
# lonW, lonE, latS, latN = -134, -59, 23, 50 # for the inner 48 states
lonW, lonE, latS, latN = -180, -30, 0, 70 # to include alaska

# define the folder path
plot_dir = Path("/place/to/store/plots")

## 3.) Load ERA5 Data & Compute Derived Fields

PV and Z are loaded at the selected pressure level and at the selected spatial domain. Two derived fields are computed:

- **Geopotential height (dam):** converted from raw geopotential (m²/s²) using MetPy's `geopotential_to_height`, then unit-converted to decameters for standard upper-air display
- **PV**: ERA5 stores PV in native SI units (m² s⁻¹ K kg⁻¹), which is multiplied by 10^6 to convert to **PVU** for plotting

In [ ]:
# load datasets from local GRIB files at selected pressure level
dsPV = get_era5_ds("PV", Year, vlevel)
dsZ = get_era5_ds("Z", Year, vlevel)

# subset variables (note: no +360 offset needed, local GRIB uses -180 to 180)
PV = dsPV['pv'].sel(time=dateTime, latitude=slice(latN+5, latS-5), longitude=slice(lonW-5, lonE+5))
Z = dsZ['z'].sel(time=dateTime, latitude=slice(latN+5, latS-5), longitude=slice(lonW-5, lonE+5))

# coordinate arrays
lats = Z.latitude
lons = Z.longitude

# convert geopotential to height (dam) and calculate potential vorticity
HGHT = mpcalc.geopotential_to_height(Z).metpy.convert_units('dam')
PV_pvu = PV * 1e6  # Convert to PVU (1 PVU = 1×10⁻⁶ m² s⁻¹ K kg⁻¹)

print(f"Height range: {float(HGHT.min()):.0f}–{float(HGHT.max()):.0f} dam")

## 4.) Contour Level Lookup Tables

Geopotential height contour intervals vary significantly by pressure level. These lookup tables define `(min, max, interval)` for each supported 
level, so the correct intervals are selected automatically based on `V_LEVEL`.

Adding a new pressure level requires only a single entry in each dict.

In [ ]:
# geopotential Height contour intervals by pressure level (dam)
HGHT_levels = {
    100:  (1560, 1680, 10),
    200:  (1140, 1260, 10),
    250:  (990,  1110, 10),
    300:  (840,  960,  10),
    500:  (504,  600,  6),
    700:  (270,  330,  3),
    850:  (90,   180,  3),
    925:  (15,   90,   3),
}

# unpack intervals for the selected level
minHGHT, maxHGHT, cint = HGHT_levels[vlevel]

HGHTcintervals = np.arange(minHGHT, maxHGHT, cint)
PVcintervals = np.arange(0, 6, 0.5)  # 2PVU tropopause

## 5.) Upper-Level Wind Speed & Geopotential Height Plot

Three overlapping layers are plotted:

1. **PV filled contours** (`BuPu`): the gradient reveals the tropopause gradient zone
2. **Geopotential height contours** (black): troughs and ridges that organize upper-level flow (decameters)
3. **2 PVU contour** (red): dynamical tropopause; where this contour dips equatorward and to low levels, it marks a **tropopause fold** and potential stratospheric intrusion, 
   a key upper-level forcing mechanism for surface cyclogenesis

In [ ]:
# set up map projections
proj_map = ccrs.LambertConformal(central_longitude=-100, central_latitude=30)
proj_data = ccrs.PlateCarree()  # Lat-lon data projection

# PLOT
fig, ax = plt.subplots(subplot_kw={'projection': proj_map}, figsize=(12, 8))

# set extent
#ax.set_extent([-127, -67, 23, 50], crs=ccrs.PlateCarree()) # for the inner 48 states
ax.set_extent([-160, -67, 25, 60], crs=ccrs.PlateCarree()) # to include alaska
# ax.set_extent([-105, -80, 35, 50], crs=ccrs.PlateCarree()) # midwest

# political/geographic boundaries
ax.coastlines(resolution='50m', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linestyle='--', linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')

# lon/lat gridlines
gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.7, linestyle='--')
gridlines.top_labels   = True
gridlines.right_labels = False
gridlines.xlabel_style = {'size': 12, 'color': 'black'}
gridlines.ylabel_style = {'size': 15, 'color': 'black'}

# 1. PV filled contours
pv_fill = ax.contourf(
    lons, lats, PV_pvu,
    levels=PVcintervals,
    cmap='BuPu', 
    transform=proj_data, 
    extend='both', zorder=1
)

# 2. geopotential height contours and labels
contour = ax.contour(
    lons, lats, HGHT, 
    levels=HGHTcintervals, 
    colors='black', linewidths=.8, 
    linestyles='solid', 
    transform=proj_data, zorder=2
)
ax.clabel(contour, inline=True, fmt='%d', fontsize=10, colors='black', inline_spacing=3)

# 3. highlight the 2 PVU dynamic tropopause
tropopause = ax.contour(
    lons, lats, PV_pvu,
    levels=[2.0],
    colors='red',
    linewidths=1.5,
    linestyles='solid',
    transform=proj_data,
    zorder=3
)
ax.clabel(tropopause, inline=True, fmt='2 PVU', fontsize=9, colors='red')

# colorbar for PV
cbar = plt.colorbar(pv_fill, ax=ax, orientation='vertical', 
                    pad=0.04, aspect=30, shrink=0.8, extendrect=True)
cbar.set_label('Potential Vorticity (PVU)', fontsize=12)
cbar.ax.tick_params(labelsize=10)

# title
title = f"ERA-5 {levelStr} hPa Potential Vorticity and Geopotential Heights (dam)"
subtitle = f"Valid at: {timeStr}"
plt.title(f"{title}\n{subtitle}", fontsize=16, fontweight='bold', loc='center')

# save and show
plt.tight_layout()

plot_path = plot_dir / f"PV_({timeStr}).png"
plt.savefig(plot_path, dpi=100)
print(f"Plot saved at: {plot_path}")

plt.show()